# Step 1. Data Loading (Should be cited)

In [2]:
from datetime import datetime
from distutils.util import strtobool

import pandas as pd
import torch

# Converts the contents in a .tsf file into a dataframe and returns it along with other meta-data of the dataset: frequency, horizon, whether the dataset contains missing values and whether the series have equal lengths
#
# Parameters
# full_file_path_and_name - complete .tsf file path
# replace_missing_vals_with - a term to indicate the missing values in series in the returning dataframe
# value_column_name - Any name that is preferred to have as the name of the column containing series values in the returning dataframe
def convert_tsf_to_dataframe(
    full_file_path_and_name,
    replace_missing_vals_with="NaN",
    value_column_name="series_value",
):
    col_names = []
    col_types = []
    all_data = {}
    line_count = 0
    frequency = None
    forecast_horizon = None
    contain_missing_values = None
    contain_equal_length = None
    found_data_tag = False
    found_data_section = False
    started_reading_data_section = False

    with open(full_file_path_and_name, "r", encoding="cp1252") as file:
        for line in file:
            # Strip white space from start/end of line
            line = line.strip()

            if line:
                if line.startswith("@"):  # Read meta-data
                    if not line.startswith("@data"):
                        line_content = line.split(" ")
                        if line.startswith("@attribute"):
                            if (
                                len(line_content) != 3
                            ):  # Attributes have both name and type
                                raise Exception("Invalid meta-data specification.")

                            col_names.append(line_content[1])
                            col_types.append(line_content[2])
                        else:
                            if (
                                len(line_content) != 2
                            ):  # Other meta-data have only values
                                raise Exception("Invalid meta-data specification.")

                            if line.startswith("@frequency"):
                                frequency = line_content[1]
                            elif line.startswith("@horizon"):
                                forecast_horizon = int(line_content[1])
                            elif line.startswith("@missing"):
                                contain_missing_values = bool(
                                    strtobool(line_content[1])
                                )
                            elif line.startswith("@equallength"):
                                contain_equal_length = bool(strtobool(line_content[1]))

                    else:
                        if len(col_names) == 0:
                            raise Exception(
                                "Missing attribute section. Attribute section must come before data."
                            )

                        found_data_tag = True
                elif not line.startswith("#"):
                    if len(col_names) == 0:
                        raise Exception(
                            "Missing attribute section. Attribute section must come before data."
                        )
                    elif not found_data_tag:
                        raise Exception("Missing @data tag.")
                    else:
                        if not started_reading_data_section:
                            started_reading_data_section = True
                            found_data_section = True
                            all_series = []

                            for col in col_names:
                                all_data[col] = []

                        full_info = line.split(":")

                        if len(full_info) != (len(col_names) + 1):
                            raise Exception("Missing attributes/values in series.")

                        series = full_info[len(full_info) - 1]
                        series = series.split(",")

                        if len(series) == 0:
                            raise Exception(
                                "A given series should contains a set of comma separated numeric values. At least one numeric value should be there in a series. Missing values should be indicated with ? symbol"
                            )

                        numeric_series = []

                        for val in series:
                            if val == "?":
                                numeric_series.append(replace_missing_vals_with)
                            else:
                                numeric_series.append(float(val))

                        if numeric_series.count(replace_missing_vals_with) == len(
                            numeric_series
                        ):
                            raise Exception(
                                "All series values are missing. A given series should contains a set of comma separated numeric values. At least one numeric value should be there in a series."
                            )

                        all_series.append(pd.Series(numeric_series).array)

                        for i in range(len(col_names)):
                            att_val = None
                            if col_types[i] == "numeric":
                                att_val = int(full_info[i])
                            elif col_types[i] == "string":
                                att_val = str(full_info[i])
                            elif col_types[i] == "date":
                                att_val = datetime.strptime(
                                    full_info[i], "%Y-%m-%d %H-%M-%S"
                                )
                            else:
                                raise Exception(
                                    "Invalid attribute type."
                                )  # Currently, the code supports only numeric, string and date types. Extend this as required.

                            if att_val is None:
                                raise Exception("Invalid attribute value.")
                            else:
                                all_data[col_names[i]].append(att_val)

                line_count = line_count + 1

        if line_count == 0:
            raise Exception("Empty file.")
        if len(col_names) == 0:
            raise Exception("Missing attribute section.")
        if not found_data_section:
            raise Exception("Missing series information under data section.")

        all_data[value_column_name] = all_series
        loaded_data = pd.DataFrame(all_data)

        return (
            loaded_data,
            frequency,
            forecast_horizon,
            contain_missing_values,
            contain_equal_length,
        )


# Example of usage
loaded_data, frequency, forecast_horizon, contain_missing_values, contain_equal_length = convert_tsf_to_dataframe("australian_electricity_demand_dataset.tsf")

print(loaded_data)
print(frequency)
print(forecast_horizon)
print(contain_missing_values)
print(contain_equal_length)

  series_name state start_timestamp  \
0          T1   NSW      2002-01-01   
1          T2   VIC      2002-01-01   
2          T3   QUN      2002-01-01   
3          T4    SA      2002-01-01   
4          T5   TAS      2002-01-01   

                                        series_value  
0  [5714.045004, 5360.189078, 5014.835118, 4602.7...  
1  [3535.867064, 3383.499028, 3655.527552, 3510.4...  
2  [3382.041342, 3288.315794, 3172.329022, 3020.3...  
3  [1191.078014, 1219.589472, 1119.173498, 1016.4...  
4  [315.915504, 306.245864, 305.762576, 295.60219...  
half_hourly
None
False
False


In [ ]:
# optional: save as xlsx
loaded_data, frequency, forecast_horizon, contain_missing_values, contain_equal_length = \
    convert_tsf_to_dataframe("australian_electricity_demand_dataset.tsf")

loaded_data.to_excel("australian_electricity_demand_dataset.xlsx", index=False, engine="openpyxl")

# Step 2. Data Preprocessing

In [3]:
# 1. Access the first time series row
series_row = loaded_data.iloc[0]

# 2. Extract its metadata
start_date = series_row["start_timestamp"]  # E.g., 2002-01-01 00:00:00
series_values = pd.Series(series_row["series_value"])

# 3. Reconstruct the full hourly DatetimeIndex for this series
full_index = pd.date_range(
    start=start_date, periods=len(series_values), freq="30min"
)

series_values.index = full_index

# 4. Slice the period and downsample from half-hourly to hourly using the mean
target_period = (
    series_values.loc["2015-02-02 01:00":"2015-02-10 00:30"]
    .resample("h")
    .mean()
)

print(target_period)
print(f"Total data points: {len(target_period)}")  # Outputs: 192

2015-02-02 01:00:00    4341.345341
2015-02-02 02:00:00    4150.155694
2015-02-02 03:00:00    4175.505814
2015-02-02 04:00:00    4573.827429
2015-02-02 05:00:00    5464.093867
                          ...     
2015-02-09 20:00:00    6766.861200
2015-02-09 21:00:00    6377.209133
2015-02-09 22:00:00    6062.101963
2015-02-09 23:00:00    5689.940227
2015-02-10 00:00:00    5301.175125
Freq: h, Length: 192, dtype: float64
Total data points: 192


In [17]:
# 5. Split the 192 points into 168 context points and 24 horizon points
context = target_period.iloc[:168]  # 7 days
horizon_true = target_period.iloc[168:]  # 1 day (for evaluation)

# 6. Convert pandas values to a PyTorch tensor and add a batch dimension
# Shape changes from (168,) to (1, 168)
context_tensor = torch.tensor(context.values, dtype=torch.float32).unsqueeze(0)

# 7. check missing or infinite values
assert not context.isna().any(), "Context contains NaN values!"